In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_timestamp, hour, when, avg, count, sum as spark_sum
from pyspark.sql.types import DoubleType, IntegerType

# Initialize Spark with 4GB Memory
spark = SparkSession.builder \
    .appName("MoneyLaunderingETLPipeline") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

print("✅ SparkSession created successfully! Version:", spark.version)
print("✅ Memory Limit Set To:", spark.conf.get("spark.driver.memory"))
spark

✅ SparkSession created successfully! Version: 3.4.1
✅ Memory Limit Set To: 4g


In [2]:
# ===============================
# LOAD DATA SOURCES (Fixed Paths)
# ===============================

base_path = "C:/Users/hp/Desktop/ess_faiss_index"  # Your main data folder

# 1. Parquet (PaySim) - already using full path, good!
df_paysim = spark.read.parquet(f"{base_path}/fraud_detection.parquet")

# 2. CSV (AML Transactions) - FIX: use full path
df_aml = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{base_path}/HI-Small_Trans.csv")
)

# 3. JSON (Currency Dictionary) - FIX: use full path
df_currency = (
    spark.read
    .option("multiline", "true")
    .json(f"{base_path}/currency.json")
)

# Check row counts
print("PaySim:", df_paysim.count())
print("AML:", df_aml.count())
print("Currency:", df_currency.count())

PaySim: 6362620
AML: 5078345
Currency: 1


In [3]:
from pyspark.sql.functions import col, explode, from_json

# ===============================
# 1. LOAD DATA SOURCES
# ===============================

# --- A. Parquet (PaySim) ---
df_paysim = spark.read.parquet("C:/Users/hp/Desktop/ess_faiss_index/fraud_detection.parquet")

# --- B. CSV (AML Transactions) ---
df_aml = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("C:/Users/hp/Desktop/ess_faiss_index/HI-Small_Trans.csv")
)

# --- C. JSON (Currency Dictionary) ---
# 1. Read the file as a single text string
df_json_raw = spark.read.option("wholetext", True).text("C:/Users/hp/Desktop/ess_faiss_index/currency.json")

# 2. Parse the text as a Map and EXPLODE it into rows
df_currency = df_json_raw.select(
    explode(
        from_json(col("value"), "MAP<STRING, STRING>")
    ).alias("Currency_Code", "Currency_Name")
)

# ===============================
# 2. SHOW RESULTS
# ===============================

print(f"PaySim Total Rows:   {df_paysim.count()}")
print(f"AML Total Rows:      {df_aml.count()}")
print(f"Currency Total Rows: {df_currency.count()}")
# As long as this says 285, your data is loaded correctly.

print("\n--- PaySim Data (Top 5) ---")
df_paysim.show(5)

print("\n--- AML Data (Top 5) ---")
df_aml.show(5)

print("\n--- Currency Data (Sorted A-Z, Top 5) ---")
# Sorting ensures you see the first currencies (starting with A)
df_currency.orderBy("Currency_Code").show(5, truncate=False)

PaySim Total Rows:   6362620
AML Total Rows:      5078345
Currency Total Rows: 286

--- PaySim Data (Top 5) ---
+----+--------+--------+-----------+-------------+--------------+-----------+--------------+--------------+-------+--------------+
|step|    type|  amount|   nameOrig|oldbalanceOrg|newbalanceOrig|   nameDest|oldbalanceDest|newbalanceDest|isFraud|isFlaggedFraud|
+----+--------+--------+-----------+-------------+--------------+-----------+--------------+--------------+-------+--------------+
|   1| PAYMENT| 9839.64|C1231006815|     170136.0|     160296.36|M1979787155|           0.0|           0.0|      0|             0|
|   1| PAYMENT| 1864.28|C1666544295|      21249.0|      19384.72|M2044282225|           0.0|           0.0|      0|             0|
|   1|TRANSFER|   181.0|C1305486145|        181.0|           0.0| C553264065|           0.0|           0.0|      1|             0|
|   1|CASH_OUT|   181.0| C840083671|        181.0|           0.0|  C38997010|       21182.0|          

In [4]:
from pyspark.sql.functions import col, explode, from_json

# ===============================
# 1. LOAD DATA SOURCES
# ===============================

# --- A. Parquet (PaySim) ---
df_paysim = spark.read.parquet("C:/Users/hp/Desktop/ess_faiss_index/fraud_detection.parquet")

# --- B. CSV (AML Transactions) ---
df_aml = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("C:/Users/hp/Desktop/ess_faiss_index/HI-Small_Trans.csv")
)

# --- C. JSON (Currency Dictionary) ---
# 1. Read the file as a single text string
df_json_raw = spark.read.option("wholetext", True).text("C:/Users/hp/Desktop/ess_faiss_index/currency.json")

# 2. Parse the text as a Map and EXPLODE it into rows
df_currency = df_json_raw.select(
    explode(
        from_json(col("value"), "MAP<STRING, STRING>")
    ).alias("Currency_Code", "Currency_Name")
)

# ===============================
# 2. SHOW RESULTS
# ===============================

print(f"PaySim Total Rows:   {df_paysim.count()}")
print(f"AML Total Rows:      {df_aml.count()}")
print(f"Currency Total Rows: {df_currency.count()}")
# As long as this says 285, your data is loaded correctly.

print("\n--- PaySim Data (Top 5) ---")
df_paysim.show(5)

print("\n--- AML Data (Top 5) ---")
df_aml.show(5)

print("\n--- Currency Data (Sorted A-Z, Top 5) ---")
# Sorting ensures you see the first currencies (starting with A)
df_currency.orderBy("Currency_Code").show(5, truncate=False)

PaySim Total Rows:   6362620
AML Total Rows:      5078345
Currency Total Rows: 286

--- PaySim Data (Top 5) ---
+----+--------+--------+-----------+-------------+--------------+-----------+--------------+--------------+-------+--------------+
|step|    type|  amount|   nameOrig|oldbalanceOrg|newbalanceOrig|   nameDest|oldbalanceDest|newbalanceDest|isFraud|isFlaggedFraud|
+----+--------+--------+-----------+-------------+--------------+-----------+--------------+--------------+-------+--------------+
|   1| PAYMENT| 9839.64|C1231006815|     170136.0|     160296.36|M1979787155|           0.0|           0.0|      0|             0|
|   1| PAYMENT| 1864.28|C1666544295|      21249.0|      19384.72|M2044282225|           0.0|           0.0|      0|             0|
|   1|TRANSFER|   181.0|C1305486145|        181.0|           0.0| C553264065|           0.0|           0.0|      1|             0|
|   1|CASH_OUT|   181.0| C840083671|        181.0|           0.0|  C38997010|       21182.0|          

In [5]:
from pyspark.sql.functions import col, explode, from_json, to_timestamp

# ==========================================
# 1. PAYSIM (Load & Standardize)
# ==========================================
df_paysim = spark.read.parquet("C:/Users/hp/Desktop/ess_faiss_index/fraud_detection.parquet")

# Rename columns to snake_case directly in df_paysim
df_paysim = (
    df_paysim
    .withColumnRenamed("nameOrig", "name_orig")
    .withColumnRenamed("oldbalanceOrg", "old_balance_org")
    .withColumnRenamed("newbalanceOrig", "new_balance_orig")
    .withColumnRenamed("nameDest", "name_dest")
    .withColumnRenamed("oldbalanceDest", "old_balance_dest")
    .withColumnRenamed("newbalanceDest", "new_balance_dest")
    .withColumnRenamed("isFraud", "is_fraud")
    .withColumnRenamed("isFlaggedFraud", "is_flagged_fraud")
)

# ==========================================
# 2. AML (Load & Standardize)
# ==========================================
df_aml = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("C:/Users/hp/Desktop/ess_faiss_index/HI-Small_Trans.csv")
)

# Rename columns and fix Timestamp directly in df_aml
df_aml = (
    df_aml
    .withColumnRenamed("Timestamp", "timestamp")
    .withColumnRenamed("From Bank", "from_bank")
    .withColumnRenamed("Account2", "from_account")
    .withColumnRenamed("To Bank", "to_bank")
    .withColumnRenamed("Account4", "to_account")
    .withColumnRenamed("Amount Received", "amount_received")
    .withColumnRenamed("Receiving Currency", "receiving_currency")
    .withColumnRenamed("Amount Paid", "amount_paid")
    .withColumnRenamed("Payment Currency", "payment_currency")
    .withColumnRenamed("Payment Format", "payment_format")
    .withColumnRenamed("Is Laundering", "is_laundering")
    .withColumn("timestamp", to_timestamp("timestamp", "yyyy/MM/dd HH:mm"))
)

# ==========================================
# 3. CURRENCY (Load & Standardize)
# ==========================================
# Read raw text, parse JSON, and save directly to df_currency
df_currency = (
    spark.read.option("wholetext", True).text("C:/Users/hp/Desktop/ess_faiss_index/currency.json")
    .select(
        explode(
            from_json(col("value"), "MAP<STRING, STRING>")
        ).alias("currency_code", "currency_name")
    )
)

# ==========================================
# CHECK RESULTS
# ==========================================
print(f"PaySim Rows:   {df_paysim.count()}")
print(f"AML Rows:      {df_aml.count()}")
print(f"Currency Rows: {df_currency.count()}")

print("\n--- PaySim (Top 5) ---")
df_paysim.show(5)

print("\n--- AML (Top 5) ---")
df_aml.show(5)

print("\n--- Currency (Top 5) ---")
df_currency.orderBy("currency_code").show(5, truncate=False)

PaySim Rows:   6362620
AML Rows:      5078345
Currency Rows: 286

--- PaySim (Top 5) ---
+----+--------+--------+-----------+---------------+----------------+-----------+----------------+----------------+--------+----------------+
|step|    type|  amount|  name_orig|old_balance_org|new_balance_orig|  name_dest|old_balance_dest|new_balance_dest|is_fraud|is_flagged_fraud|
+----+--------+--------+-----------+---------------+----------------+-----------+----------------+----------------+--------+----------------+
|   1| PAYMENT| 9839.64|C1231006815|       170136.0|       160296.36|M1979787155|             0.0|             0.0|       0|               0|
|   1| PAYMENT| 1864.28|C1666544295|        21249.0|        19384.72|M2044282225|             0.0|             0.0|       0|               0|
|   1|TRANSFER|   181.0|C1305486145|          181.0|             0.0| C553264065|             0.0|             0.0|       1|               0|
|   1|CASH_OUT|   181.0| C840083671|          181.0|       

In [6]:
from pyspark.sql.functions import col, count, when, isnan

# ==========================================
# FUNCTION: Check Nulls for a DataFrame
# ==========================================
def check_missing_values(df, df_name):
    print(f"\n--- Missing Values Report: {df_name} ---")
    
    # Select count of nulls for every column
    df.select([
        count(when(col(c).isNull() | isnan(c), c)).alias(c) 
        for c in df.columns 
        if dict(df.dtypes)[c] in ['int', 'double', 'float', 'long'] # Check NaNs only for numbers
    ] + [
        count(when(col(c).isNull(), c)).alias(c) 
        for c in df.columns 
        if dict(df.dtypes)[c] not in ['int', 'double', 'float', 'long'] # Check Nulls for others
    ]).show(truncate=False, vertical=True)

# ==========================================
# EXECUTE CHECKS
# ==========================================

# 1. Check PaySim
check_missing_values(df_paysim, "PaySim Data")

# 2. Check AML
check_missing_values(df_aml, "AML Data")

# 3. Check Currency
check_missing_values(df_currency, "Currency Data")


--- Missing Values Report: PaySim Data ---
-RECORD 0---------------
 amount           | 0   
 old_balance_org  | 0   
 new_balance_orig | 0   
 old_balance_dest | 0   
 new_balance_dest | 0   
 step             | 0   
 type             | 0   
 name_orig        | 0   
 name_dest        | 0   
 is_fraud         | 0   
 is_flagged_fraud | 0   


--- Missing Values Report: AML Data ---
-RECORD 0-----------------
 from_bank          | 0   
 to_bank            | 0   
 amount_received    | 0   
 amount_paid        | 0   
 is_laundering      | 0   
 timestamp          | 0   
 from_account       | 0   
 to_account         | 0   
 receiving_currency | 0   
 payment_currency   | 0   
 payment_format     | 0   


--- Missing Values Report: Currency Data ---
-RECORD 0------------
 currency_code | 0   
 currency_name | 0   



In [7]:
from pyspark.sql.functions import col

# ==========================================
# 1. ENRICHMENT (Join Operation) - FIXED
# ==========================================
# Logic: The AML data has names ("US Dollar"), but we want standard codes ("USD").
# We join on the NAME to retrieve the CODE.

df_aml_enriched = df_aml.join(
    df_currency.alias("curr"), 
    df_aml.payment_currency == col("curr.currency_name"), # Match "US Dollar" to "US Dollar"
    "left"
).select(
    df_aml["*"], 
    col("curr.currency_code").alias("standard_currency_code") # We get the Code "USD" back
)

print("✅ STEP 1 COMPLETE: Added Standard Currency Code.")
# showing where the code is NOT null to prove it works
df_aml_enriched.filter(col("standard_currency_code").isNotNull()) \
    .select("payment_currency", "standard_currency_code", "amount_paid") \
    .show(5, truncate=False)

✅ STEP 1 COMPLETE: Added Standard Currency Code.
+----------------+----------------------+-----------+
|payment_currency|standard_currency_code|amount_paid|
+----------------+----------------------+-----------+
|US Dollar       |USD                   |3697.34    |
|US Dollar       |USD                   |0.01       |
|US Dollar       |USD                   |14675.57   |
|US Dollar       |USD                   |2806.97    |
|US Dollar       |USD                   |36682.97   |
+----------------+----------------------+-----------+
only showing top 5 rows



In [25]:
from pyspark.sql.functions import col, hour, date_format, when, count, sum, round
from pyspark.sql.window import Window

# ======================================================
# STEP 1: ENRICHMENT (Join with Currency Dictionary)
# ======================================================
print("⏳ Executing Step 1: Joining AML data with Currency Dictionary...")

# Logic: Match "US Dollar" (AML) -> "US Dollar" (Currency) to get "USD" (Code)
# df_aml_enriched = df_aml.join(
#     df_currency.alias("curr"), 
#     df_aml.payment_currency == col("curr.currency_name"), 
#     "left"
# ).select(
#     df_aml["*"], 
#     col("curr.currency_code").alias("standard_currency_code") 
# )

# ======================================================
# STEP 2: FEATURE ENGINEERING (Time, Money & Volume)
# ======================================================
print("⏳ Executing Step 2: Calculating Advanced Features & Normalizing...")

# A. Define Window per Bank (To calculate total volume per bank)
bank_window = Window.partitionBy("from_bank")

# B. Apply Transformations
df_aml_transformed = df_aml_enriched \
    .withColumn("hour", hour("timestamp")) \
    .withColumn("day_name", date_format("timestamp", "EEEE")) \
    \
    .withColumn(
        "transaction_type",
        when(col("amount_paid") < 500, "Small")
        .when((col("amount_paid") >= 500) & (col("amount_paid") < 10000), "Medium")
        .otherwise("Large / High Value")
    ) \
    \
    .withColumn(
        "bank_activity_count", 
        count("timestamp").over(bank_window)
    ) \
    .withColumn(
        "bank_total_money_raw", 
        sum("amount_paid").over(bank_window)
    ) \
    .withColumn(
        "amount_in_billions", 
        round(col("amount_paid") / 1000000000, 2)
    ) \
    .withColumn(
        "bank_total_billions", 
        round(col("bank_total_money_raw") / 1000000000, 2)
    )

# ======================================================
# FINAL OUTPUT
# ======================================================
print("\n✅ TRANSFORMATION COMPLETE: Data Enriched & Engineered.")

print("\n--- Final Data Preview (Sorted by Amount) ---")
df_aml_transformed.select(
    "timestamp", 
    "day_name", 
    "standard_currency_code",    # From Step 1 (Join)
    "amount_in_billions",        # From Step 2 (Normalization)
    "bank_total_billions",       # From Step 2 (Window Function)
    "transaction_type"           # From Step 2 (Bucketing)
).orderBy(col("amount_in_billions").desc()).show(10, truncate=False)

⏳ Executing Step 1: Joining AML data with Currency Dictionary...
⏳ Executing Step 2: Calculating Advanced Features & Normalizing...

✅ TRANSFORMATION COMPLETE: Data Enriched & Engineered.

--- Final Data Preview (Sorted by Amount) ---
+-------------------+---------+----------------------+------------------+-------------------+------------------+
|timestamp          |day_name |standard_currency_code|amount_in_billions|bank_total_billions|transaction_type  |
+-------------------+---------+----------------------+------------------+-------------------+------------------+
|2022-09-06 17:49:00|Tuesday  |JPY                   |1046.3            |1123.37            |Large / High Value|
|2022-09-07 08:30:00|Wednesday|INR                   |965.93            |1165.08            |Large / High Value|
|2022-09-02 18:56:00|Friday   |RUB                   |626.04            |1509.86            |Large / High Value|
|2022-09-09 08:37:00|Friday   |RUB                   |626.04            |1509.86       

In [9]:
from pyspark.sql.functions import avg, col, when
from pyspark.sql.window import Window

# ==========================================
# 3. ADVANCED ANALYTICS (Window Functions)
# ==========================================
window_spec = Window.partitionBy("name_orig")

df_paysim_transformed = df_paysim.withColumn(
    "avg_amount_per_user", 
    avg("amount").over(window_spec)
).withColumn(
    "deviation_from_avg", 
    col("amount") - col("avg_amount_per_user")
).withColumn(
    "risk_category",
    when(col("amount") > 200000, "High Risk")
    .when(col("amount") > 10000, "Medium Risk")
    .otherwise("Low Risk")
)

print("✅ STEP 3 COMPLETE: Risk Bucketing Applied.")
df_paysim_transformed.select("name_orig", "amount", "avg_amount_per_user", "risk_category").show(5)

✅ STEP 3 COMPLETE: Risk Bucketing Applied.
+-----------+--------+-------------------+-------------+
|  name_orig|  amount|avg_amount_per_user|risk_category|
+-----------+--------+-------------------+-------------+
|C1000001337| 3170.28|            3170.28|     Low Risk|
|C1000003615|49360.77|           49360.77|  Medium Risk|
|C1000006873| 4615.66|            4615.66|     Low Risk|
|C1000011070|72459.52|           72459.52|  Medium Risk|
|C1000018730| 1182.35|            1182.35|     Low Risk|
+-----------+--------+-------------------+-------------+
only showing top 5 rows



In [10]:
from pyspark.sql.functions import col, count, when, isnan, trim

# ==========================================
# FUNCTION: Check Nulls, NaNs, AND Empty Strings
# ==========================================
def check_full_data_quality(df, df_name):
    print(f"\n==========================================")
    print(f"📊 FULL QUALITY CHECK: {df_name}")
    print(f"   (Checking Nulls, NaNs, and Empty Strings)")
    print(f"==========================================")
    
    select_exprs = []
    
    for c in df.columns:
        dtype = dict(df.dtypes)[c]
        
        # 1. NUMERIC COLUMNS (Int, Double, Float)
        # Check for NULL or NaN
        if dtype in ['int', 'double', 'float', 'long']:
            condition = col(c).isNull() | isnan(col(c))
            
        # 2. STRING COLUMNS
        # Check for NULL, Empty String "", or text "NaN"/"null"
        elif dtype == 'string':
            condition = (
                col(c).isNull() | 
                (trim(col(c)) == "") | 
                (col(c) == "NaN") |
                (col(c) == "null")
            )
            
        # 3. OTHER (Timestamp, Boolean)
        # Check for NULL
        else:
            condition = col(c).isNull()
            
        # Create the count for this column
        select_exprs.append(count(when(condition, c)).alias(c))

    # Run the query and show results vertically
    # TARGET: You want to see '0' for every column.
    df.select(select_exprs).show(truncate=False, vertical=True)

# ==========================================
# EXECUTE THE CHECKS
# ==========================================

# 1. Check AML Data
check_full_data_quality(df_aml_transformed, "AML Final Data")

# 2. Check PaySim Data
check_full_data_quality(df_paysim_transformed, "PaySim Final Data")


📊 FULL QUALITY CHECK: AML Final Data
   (Checking Nulls, NaNs, and Empty Strings)
-RECORD 0---------------------
 timestamp              | 0   
 from_bank              | 0   
 from_account           | 0   
 to_bank                | 0   
 to_account             | 0   
 amount_received        | 0   
 receiving_currency     | 0   
 amount_paid            | 0   
 payment_currency       | 0   
 payment_format         | 0   
 is_laundering          | 0   
 standard_currency_code | 0   
 hour                   | 0   
 day_name               | 0   
 transaction_type       | 0   
 bank_activity_count    | 0   
 bank_total_money_raw   | 0   
 amount_in_billions     | 0   
 bank_total_billions    | 0   


📊 FULL QUALITY CHECK: PaySim Final Data
   (Checking Nulls, NaNs, and Empty Strings)
-RECORD 0------------------
 step                | 0   
 type                | 0   
 amount              | 0   
 name_orig           | 0   
 old_balance_org     | 0   
 new_balance_orig    | 0   
 name_dest    

In [11]:
# ======================================================
# FULL DATA INSPECTION (All Columns)
# ======================================================

print("📊 1. AML DATA (Full Structure)")
print("   (Sorted by Amount so you see the Calculated Billions & Money Types)")
# Showing top 3 rows, truncating long strings to keep it readable on screen
df_aml_transformed.orderBy(col("amount_in_billions").desc()).show(3)


print("\n📊 2. PAYSIM DATA (Full Structure)")
print("   (Filtered for 'High Risk' so you see the Risk Categories)")
# Showing top 3 rows
df_paysim_transformed.filter(col("risk_category") == "High Risk").show(3)

📊 1. AML DATA (Full Structure)
   (Sorted by Amount so you see the Calculated Billions & Money Types)
+-------------------+---------+------------+-------+----------+-------------------+------------------+-------------------+----------------+--------------+-------------+----------------------+----+---------+------------------+-------------------+--------------------+------------------+-------------------+
|          timestamp|from_bank|from_account|to_bank|to_account|    amount_received|receiving_currency|        amount_paid|payment_currency|payment_format|is_laundering|standard_currency_code|hour| day_name|  transaction_type|bank_activity_count|bank_total_money_raw|amount_in_billions|bank_total_billions|
+-------------------+---------+------------+-------+----------+-------------------+------------------+-------------------+----------------+--------------+-------------+----------------------+----+---------+------------------+-------------------+--------------------+------------------+-

In [12]:
df_aml_with_risk = df_aml_transformed.withColumn(
    "risk_category",
    when(col("transaction_type") == "Small Value", "Low Risk")
    .when(col("transaction_type") == "Medium Value", "Medium Risk")
    .otherwise("High Risk")
)


In [13]:
from pyspark.sql.functions import count, avg, sum

df_paysim_agg = df_paysim_transformed.groupBy("risk_category").agg(
    count("*").alias("paysim_txn_count"),
    avg("amount").alias("avg_fraud_amount"),
    sum("is_fraud").alias("fraud_cases")
)


In [14]:
df_connected = df_aml_with_risk.join(
    df_paysim_agg,
    on="risk_category",
    how="left"
)


In [15]:
from pyspark.sql.functions import col, lit

df_connected = (
    df_connected
    .fillna({
        "avg_fraud_amount": 0.0,
        "fraud_cases": 0
    })
)


In [16]:
from pyspark.sql.functions import when

df_connected = df_connected.withColumn(
    "paysim_fraud_present",
    when(col("fraud_cases") > 0, 1).otherwise(0)
)


In [17]:
df_connected = df_connected.withColumn(
    "is_night_transaction",
    when((col("hour") < 6) | (col("hour") > 22), 1).otherwise(0)
)


In [18]:
df_connected = df_connected.withColumn(
    "aml_flag",
    when(
        (col("risk_category") == "High Risk") |
        (col("paysim_fraud_present") == 1) |
        (col("is_night_transaction") == 1),
        1
    ).otherwise(0)
)


In [ ]:
from pyspark.sql.functions import col, sum

# ==============================
# Count NULLs per column (df_connected)
# ==============================
null_counts = df_connected.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df_connected.columns
])

null_counts.show(truncate=False)

# ==============================
# Count total rows
# ==============================
total_rows = df_connected.count()
print(f"Total rows in df_connected: {total_rows}")
df_connected.show(5)

+-------------+---------+---------+------------+-------+----------+---------------+------------------+-----------+----------------+--------------+-------------+----------------------+----+--------+----------------+-------------------+--------------------+------------------+-------------------+----------------+----------------+-----------+--------------------+--------------------+--------+
|risk_category|timestamp|from_bank|from_account|to_bank|to_account|amount_received|receiving_currency|amount_paid|payment_currency|payment_format|is_laundering|standard_currency_code|hour|day_name|transaction_type|bank_activity_count|bank_total_money_raw|amount_in_billions|bank_total_billions|paysim_txn_count|avg_fraud_amount|fraud_cases|paysim_fraud_present|is_night_transaction|aml_flag|
+-------------+---------+---------+------------+-------+----------+---------------+------------------+-----------+----------------+--------------+-------------+----------------------+----+--------+----------------+--

In [20]:
df_final = df_connected.select(
    # Time
    "timestamp",
    "hour",
    "day_name",

    # Accounts
    "from_bank",
    "to_bank",

    # Amount
    "amount_paid",
    "payment_currency",
    "payment_format",

    # Risk logic
    "transaction_type",
    "risk_category",
    "paysim_fraud_present",
    "is_night_transaction",
    "aml_flag",

    # Label
    "is_laundering"
)


In [21]:
spark.conf.set("spark.hadoop.io.native.lib.available", "false")
spark.conf.set("spark.sql.parquet.enableVectorizedReader", "false")


In [22]:
import duckdb
import os
from pathlib import Path

duckdb_path = r"C:\Users\hp\Desktop\ess_faiss_index\analytics.duckdb"
folder_path = r"C:\Users\hp\Desktop\ess_faiss_index\df_final_parquet_new"

# Convert to Path object and check
folder = Path(folder_path)
if not folder.exists():
    print(f"Folder doesn't exist: {folder}")
    print(f"Current working directory: {os.getcwd()}")
    
# List all files
parquet_files = list(folder.glob("*.parquet"))
print(f"Found {len(parquet_files)} parquet files")

if parquet_files:
    # Create a list of file paths as strings
    file_paths = [str(f) for f in parquet_files]
    
    con = duckdb.connect(duckdb_path)
    
    # Use the list of file paths
    con.execute("""
        CREATE OR REPLACE TABLE transactions AS
        SELECT *
        FROM read_parquet(?)
    """, [file_paths])
    
    print(con.execute("SELECT COUNT(*) FROM transactions").fetchone())
    con.close()
else:
    print("No parquet files found. Check:")
    print(f"1. Folder exists: {folder.exists()}")
    print(f"2. Folder contents: {list(folder.iterdir()) if folder.exists() else 'N/A'}")

Folder doesn't exist: C:\Users\hp\Desktop\ess_faiss_index\df_final_parquet_new
Current working directory: c:\Users\hp\Desktop\ess_faiss_index
Found 0 parquet files
No parquet files found. Check:
1. Folder exists: False
2. Folder contents: N/A


In [23]:
import duckdb
import pandas as pd
from pyspark.sql.functions import col

# Path to your DuckDB file
duckdb_path = r"C:\Users\hp\Desktop\ess_faiss_index\analytics.duckdb"

# Connect to DuckDB
con = duckdb.connect(duckdb_path)

print("Loading transaction data from PySpark DataFrame to DuckDB...")

# OPTION 1: Convert timestamp columns explicitly before converting to pandas
try:
    # Check schema first
    print("DataFrame schema:")
    df_final.printSchema()
    
    # Convert timestamp column to string first to avoid pandas conversion issues
    from pyspark.sql.functions import date_format
    
    # Create a temporary DataFrame with timestamp as string
    df_temp = df_final.withColumn("timestamp_str", date_format(col("timestamp"), "yyyy-MM-dd HH:mm:ss"))
    
    # Drop the original timestamp column and keep the string version
    columns_to_keep = [c for c in df_temp.columns if c != "timestamp"]
    df_temp = df_temp.select(*columns_to_keep).withColumnRenamed("timestamp_str", "timestamp")
    
    # Now convert to pandas
    print("Converting to pandas...")
    pandas_df = df_temp.toPandas()
    
    # Convert timestamp string back to datetime in pandas
    pandas_df['timestamp'] = pd.to_datetime(pandas_df['timestamp'])
    
    print(f"Successfully converted {len(pandas_df):,} rows to pandas DataFrame")
    
except Exception as e:
    print(f"Error: {e}")
    print("\nTrying alternative approach...")
    
    # OPTION 2: Convert column by column
    pandas_df = pd.DataFrame()
    
    for column in df_final.columns:
        print(f"Converting column: {column}")
        # Get column data type
        col_type = dict(df_final.dtypes)[column]
        
        if col_type == 'timestamp':
            # For timestamp, extract as string and convert
            col_data = df_final.select(col(column)).toPandas()
            # The column name might have issues, use iloc
            pandas_df[column] = pd.to_datetime(col_data.iloc[:, 0])
        else:
            # For other columns, convert normally
            col_data = df_final.select(col(column)).toPandas()
            pandas_df[column] = col_data.iloc[:, 0]

# Load into DuckDB
print("\nLoading into DuckDB...")
con.execute("CREATE OR REPLACE TABLE transactions AS SELECT * FROM pandas_df")

# Verify
count = con.execute("SELECT COUNT(*) FROM transactions").fetchone()[0]
print(f"Successfully loaded {count:,} rows into DuckDB")

con.close()

Loading transaction data from PySpark DataFrame to DuckDB...
DataFrame schema:
root
 |-- timestamp: timestamp (nullable = true)
 |-- hour: integer (nullable = true)
 |-- day_name: string (nullable = true)
 |-- from_bank: integer (nullable = true)
 |-- to_bank: integer (nullable = true)
 |-- amount_paid: double (nullable = true)
 |-- payment_currency: string (nullable = true)
 |-- payment_format: string (nullable = true)
 |-- transaction_type: string (nullable = false)
 |-- risk_category: string (nullable = false)
 |-- paysim_fraud_present: integer (nullable = false)
 |-- is_night_transaction: integer (nullable = false)
 |-- aml_flag: integer (nullable = false)
 |-- is_laundering: integer (nullable = true)

Converting to pandas...
Successfully converted 5,078,345 rows to pandas DataFrame

Loading into DuckDB...
Successfully loaded 5,078,345 rows into DuckDB


In [24]:
import duckdb

# Path to your DuckDB file
duckdb_path = r"C:\Users\hp\Desktop\ess_faiss_index\analytics.duckdb"

# Connect to the database
con = duckdb.connect(duckdb_path)

print("=== Verifying Loaded Data ===")

# List all tables
print("\n1. Tables in the database:")
tables = con.execute("SHOW TABLES").fetchall()
for table in tables:
    table_name = table[0]
    count = con.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]
    print(f"   - {table_name}: {count:,} rows")

# Check if 'transactions' table exists
if 'transactions' in [t[0] for t in tables]:
    print("\n2. Transactions table schema:")
    schema = con.execute("DESCRIBE transactions").fetchall()
    for col in schema:
        print(f"   - {col[0]}: {col[1]}")
    
    # Sample data
    print("\n3. Sample data (first 3 rows):")
    sample = con.execute("SELECT * FROM transactions LIMIT 3").fetchall()
    for row in sample:
        print(row)
    
    # Basic statistics
    print("\n4. Basic statistics:")
    stats = con.execute("""
        SELECT 
            COUNT(*) as total_rows,
            MIN(timestamp) as earliest,
            MAX(timestamp) as latest,
            SUM(is_laundering) as fraud_cases,
            ROUND(100.0 * SUM(is_laundering) / COUNT(*), 2) as fraud_percentage
        FROM transactions
    """).fetchone()
    
    print(f"   Total rows: {stats[0]:,}")
    print(f"   Date range: {stats[1]} to {stats[2]}")
    print(f"   Fraud cases: {stats[3]:,} ({stats[4]}%)")
else:
    print("ERROR: 'transactions' table not found!")

con.close()

=== Verifying Loaded Data ===

1. Tables in the database:
   - transactions: 5,078,345 rows

2. Transactions table schema:
   - hour: INTEGER
   - day_name: VARCHAR
   - from_bank: INTEGER
   - to_bank: INTEGER
   - amount_paid: DOUBLE
   - payment_currency: VARCHAR
   - payment_format: VARCHAR
   - transaction_type: VARCHAR
   - risk_category: VARCHAR
   - paysim_fraud_present: INTEGER
   - is_night_transaction: INTEGER
   - aml_flag: INTEGER
   - is_laundering: INTEGER
   - timestamp: TIMESTAMP_NS

3. Sample data (first 3 rows):
(0, 'Thursday', 10, 10, 3697.34, 'US Dollar', 'Reinvestment', 'Medium', 'High Risk', 1, 1, 1, 0, datetime.datetime(2022, 9, 1, 0, 20))
(0, 'Thursday', 3208, 1, 0.01, 'US Dollar', 'Cheque', 'Small', 'High Risk', 1, 1, 1, 0, datetime.datetime(2022, 9, 1, 0, 20))
(0, 'Thursday', 3209, 3209, 14675.57, 'US Dollar', 'Reinvestment', 'Large / High Value', 'High Risk', 1, 1, 1, 0, datetime.datetime(2022, 9, 1, 0, 0))

4. Basic statistics:
   Total rows: 5,078,345
   D